# 09 — Generate PPTX Presentation
Assembles the PNG figures produced by notebook 08 into a 16-slide MVP PowerPoint.
Run notebook 08 first to generate all figures, then run all cells here top-to-bottom.

In [ ]:
## ── 0. Imports & Config ──────────────────────────────────────────────────────
from pathlib import Path
from pptx import Presentation
from pptx.util import Inches, Pt, Emu
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.util import Cm
import copy

project_root = Path.cwd().parent
FIG_DIR  = project_root / 'output' / 'presentation'
OUT_PPTX = project_root / 'output' / 'car_price_prediction_mvp.pptx'

# ── Theme colours ──────────────────────────────────────────────────────────────
C_DARK_BLUE = RGBColor(0x1B, 0x3A, 0x6B)   # #1B3A6B
C_MID_BLUE  = RGBColor(0x2E, 0x6D, 0xA4)   # #2E6DA4
C_ACCENT    = RGBColor(0x4A, 0x90, 0xD9)   # #4A90D9
C_WHITE     = RGBColor(0xFF, 0xFF, 0xFF)
C_LIGHT     = RGBColor(0xF4, 0xF6, 0xF8)

# ── Slide dimensions: 16:9 widescreen ────────────────────────────────────────
SLIDE_W = Inches(13.33)
SLIDE_H = Inches(7.5)

# ── Check figures dir ────────────────────────────────────────────────────────
figs = sorted(FIG_DIR.glob('*.png'))
print(f'Found {len(figs)} PNG figures in {FIG_DIR}')
for f in figs:
    print(f'  {f.name}')


In [ ]:
## ── 1. Helper Functions ──────────────────────────────────────────────────────
def new_prs():
    """Create a blank 16:9 presentation."""
    prs = Presentation()
    prs.slide_width  = SLIDE_W
    prs.slide_height = SLIDE_H
    return prs


def add_background(slide, color=C_DARK_BLUE):
    """Fill slide background with a solid colour."""
    background = slide.background
    fill = background.fill
    fill.solid()
    fill.fore_color.rgb = color


def add_text_box(slide, text, left, top, width, height,
                 font_size=18, bold=False, color=C_WHITE,
                 align=PP_ALIGN.LEFT, wrap=True):
    txBox = slide.shapes.add_textbox(left, top, width, height)
    tf    = txBox.text_frame
    tf.word_wrap = wrap
    p = tf.paragraphs[0]
    p.alignment = align
    run = p.add_run()
    run.text = text
    run.font.size = Pt(font_size)
    run.font.bold = bold
    run.font.color.rgb = color
    return txBox


def add_title_bar(slide, title_text, subtitle_text=''):
    """Dark-blue banner at the top with white title."""
    bar = slide.shapes.add_shape(
        1,  # MSO_SHAPE_TYPE.RECTANGLE
        Inches(0), Inches(0), SLIDE_W, Inches(1.2)
    )
    bar.fill.solid()
    bar.fill.fore_color.rgb = C_DARK_BLUE
    bar.line.fill.background()

    add_text_box(slide, title_text,
                 Inches(0.3), Inches(0.05), Inches(12.5), Inches(0.75),
                 font_size=28, bold=True, color=C_WHITE, align=PP_ALIGN.LEFT)
    if subtitle_text:
        add_text_box(slide, subtitle_text,
                     Inches(0.3), Inches(0.75), Inches(12.5), Inches(0.4),
                     font_size=14, bold=False, color=C_ACCENT, align=PP_ALIGN.LEFT)


def add_image_centered(slide, img_path, top=Inches(1.2),
                        max_w=Inches(12.8), max_h=Inches(6.1)):
    """Add image centered in the body area, scaling to fit."""
    from PIL import Image as PILImage
    with PILImage.open(img_path) as im:
        iw, ih = im.size
    scale = min(max_w / iw * 914400, max_h / ih * 914400)  # EMU per pixel = 914400/96
    # But pptx uses actual pixel dimensions at DPI; easier to just use Inches
    # Compute final size in inches
    dpi   = 200  # we saved at 200 dpi
    img_w_in = iw / dpi
    img_h_in = ih / dpi
    scale_x  = (max_w / 914400) / img_w_in
    scale_y  = (max_h / 914400) / img_h_in
    scale_f  = min(scale_x, scale_y)
    fw = Inches(img_w_in * scale_f)
    fh = Inches(img_h_in * scale_f)
    left = int((SLIDE_W - fw) / 2)
    top_pos = int(top + (max_h - fh) / 2)
    slide.shapes.add_picture(str(img_path), left, top_pos, fw, fh)


def content_slide(prs, title, subtitle, img_path, notes=''):
    """Standard content slide: title bar + centred figure."""
    sl_layout = prs.slide_layouts[6]  # blank layout
    slide = prs.slides.add_slide(sl_layout)
    add_background(slide, C_WHITE)
    add_title_bar(slide, title, subtitle)
    if img_path and Path(img_path).exists():
        add_image_centered(slide, img_path)
    else:
        add_text_box(slide, f'[Figure missing: {img_path}]',
                     Inches(1), Inches(2), Inches(11), Inches(4),
                     font_size=16, color=RGBColor(0xCC, 0x33, 0x33))
    if notes:
        slide.notes_slide.notes_text_frame.text = notes
    return slide


def bullet_slide(prs, title, subtitle, bullets, notes=''):
    """Slide with bullet-point body (no figure)."""
    sl_layout = prs.slide_layouts[6]
    slide = prs.slides.add_slide(sl_layout)
    add_background(slide, C_WHITE)
    add_title_bar(slide, title, subtitle)
    # body area
    body_top  = Inches(1.35)
    body_left = Inches(0.5)
    body_w    = Inches(12.3)
    body_h    = Inches(5.9)
    txBox = slide.shapes.add_textbox(body_left, body_top, body_w, body_h)
    tf = txBox.text_frame
    tf.word_wrap = True
    for i, bullet in enumerate(bullets):
        if i == 0:
            p = tf.paragraphs[0]
        else:
            p = tf.add_paragraph()
        p.text = bullet
        p.level = 0
        p.font.size = Pt(17)
        p.font.color.rgb = RGBColor(0x1A, 0x1A, 0x2E)
    if notes:
        slide.notes_slide.notes_text_frame.text = notes
    return slide


def title_slide(prs, main_title, subtitle, footer=''):
    """Cover slide with dark blue background."""
    sl_layout = prs.slide_layouts[6]
    slide = prs.slides.add_slide(sl_layout)
    add_background(slide, C_DARK_BLUE)

    # Accent bar (top)
    bar = slide.shapes.add_shape(1,
        Inches(0), Inches(0), SLIDE_W, Inches(0.12))
    bar.fill.solid(); bar.fill.fore_color.rgb = C_ACCENT
    bar.line.fill.background()

    add_text_box(slide, main_title,
                 Inches(1), Inches(1.8), Inches(11.3), Inches(1.8),
                 font_size=40, bold=True, color=C_WHITE, align=PP_ALIGN.CENTER)
    add_text_box(slide, subtitle,
                 Inches(1), Inches(3.7), Inches(11.3), Inches(1.2),
                 font_size=22, bold=False, color=C_ACCENT, align=PP_ALIGN.CENTER)
    if footer:
        add_text_box(slide, footer,
                     Inches(0.5), Inches(6.7), Inches(12.3), Inches(0.5),
                     font_size=11, color=RGBColor(0x88, 0xAA, 0xCC), align=PP_ALIGN.CENTER)
    return slide


def two_image_slide(prs, title, subtitle, img_left, img_right, notes=''):
    """Side-by-side two-image layout."""
    sl_layout = prs.slide_layouts[6]
    slide = prs.slides.add_slide(sl_layout)
    add_background(slide, C_WHITE)
    add_title_bar(slide, title, subtitle)
    for img_path, left_in in [(img_left, 0.15), (img_right, 6.65)]:
        if img_path and Path(img_path).exists():
            slide.shapes.add_picture(str(img_path),
                                     Inches(left_in), Inches(1.25),
                                     Inches(6.45), Inches(6.0))
    if notes:
        slide.notes_slide.notes_text_frame.text = notes
    return slide


print('✓ Helper functions defined')


In [ ]:
## ── 2. Build Presentation ───────────────────────────────────────────────────
try:
    from PIL import Image  # ensure Pillow is available (used in add_image_centered)
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'Pillow', '-q'])
    from PIL import Image

prs = new_prs()

# ── Helper to find a figure by prefix number ─────────────────────────────────
def fig(prefix):
    """Return first PNG whose stem starts with prefix, or None."""
    matches = sorted(FIG_DIR.glob(f'{prefix}*.png'))
    return matches[0] if matches else None

# ────────────────────────────────────────────────────────────────────────────
# SLIDE 01 — Cover
# ────────────────────────────────────────────────────────────────────────────
title_slide(
    prs,
    main_title='Used Car Price Prediction\nfor Customs Valuation',
    subtitle='LightGBM Quantile Regression — Le Boncoin France (620 K listings)',
    footer='MVP Presentation · March 2026'
)
print('✓ Slide 01 — Cover')

# ────────────────────────────────────────────────────────────────────────────
# SLIDE 02 — Agenda
# ────────────────────────────────────────────────────────────────────────────
bullet_slide(
    prs,
    title='Agenda',
    subtitle='',
    bullets=[
        '1 · Data Source — Le Boncoin France (732 K → 620 K listings)',
        '2 · Data Cleaning Pipeline',
        '3 · Feature Engineering — 43 input features',
        '4 · Model Architecture — LightGBM Quantile Regression',
        '5 · Model Performance on held-out test set',
        '6 · Feature Importance — SHAP top-20',
        '7 · Customs Application — Côte d\'Ivoire & Cameroon',
        '8 · Validation Results & Tier Classification',
    ],
    notes='Welcome. Walk through each section in order.'
)
print('✓ Slide 02 — Agenda')

# ────────────────────────────────────────────────────────────────────────────
# SLIDE 03 — Data Source
# ────────────────────────────────────────────────────────────────────────────
content_slide(
    prs,
    title='Data Source — Le Boncoin France',
    subtitle='Web scrape of 732,427 listings · October 13, 2025',
    img_path=fig('01_'),
    notes='Scraped from Le Boncoin, the largest used-car marketplace in France.\n'
          '732 K raw listings → 620 K after cleaning.\n'
          '146 raw brands → 43 brands in the model.\n'
          'Columns retained: brand, model, year, km, price.'
)
print('✓ Slide 03 — Data Source')

# ────────────────────────────────────────────────────────────────────────────
# SLIDE 04 — Cleaning Pipeline
# ────────────────────────────────────────────────────────────────────────────
content_slide(
    prs,
    title='Data Cleaning Pipeline',
    subtitle='5-step funnel: 732 K → 620 K records',
    img_path=fig('02_'),
    notes='Steps:\n'
          '1. Remove pre-1990 and "autre" brand\n'
          '2. IQR horsepower outliers\n'
          '3. Brands with < 400 listings removed (rare brands)\n'
          '4. Per-brand IQR on log(price) — multiplier 1.5\n'
          '5. Per-brand IQR on km — multiplier 1.5'
)
print('✓ Slide 04 — Cleaning Pipeline')

# ────────────────────────────────────────────────────────────────────────────
# SLIDE 05 — Feature Engineering
# ────────────────────────────────────────────────────────────────────────────
content_slide(
    prs,
    title='Feature Engineering — 43 Input Features',
    subtitle='Time · Brand statistics · Model statistics · Brand OHE · Model size',
    img_path=fig('03_'),
    notes='No raw categorical text is passed to the model.\n'
          'All categorical information is encoded as aggregated statistics or OHE.\n'
          'Brand / model stats capture price distributions in the training data.'
)
print('✓ Slide 05 — Feature Engineering')

# ────────────────────────────────────────────────────────────────────────────
# SLIDE 06 — Quantile Corridor Concept
# ────────────────────────────────────────────────────────────────────────────
content_slide(
    prs,
    title='Quantile Regression — 70% Price Corridor',
    subtitle='Three parallel LightGBM models: Q15 · Q50 · Q85',
    img_path=fig('04_'),
    notes='Instead of a single point estimate, we predict a price range.\n'
          'Q15 = lower bound (15th percentile of the market)\n'
          'Q50 = median estimate (reference price)\n'
          'Q85 = upper bound (85th percentile)\n'
          'The corridor covers ~70% of the market price distribution.'
)
print('✓ Slide 06 — Quantile Corridor')

# ────────────────────────────────────────────────────────────────────────────
# SLIDE 07 — Model Architecture (bullets)
# ────────────────────────────────────────────────────────────────────────────
bullet_slide(
    prs,
    title='Model Architecture',
    subtitle='LightGBM gradient-boosted decision trees',
    bullets=[
        'Algorithm : LGBMRegressor  ·  objective = "quantile"',
        'Target    : log(price in EUR)  →  back-transformed with exp()',
        'Quantiles : α = 0.15 (Q15)  ·  0.50 (Q50)  ·  0.85 (Q85)',
        'Hyperparameters : n_estimators = 5 000  ·  learning_rate = 0.10',
        'Train / Test split : 80 % / 20 %  ·  random_state = 42',
        'Feature matrix : 43 columns  ·  No raw categorical strings',
        'Inference on customs data : km = 100 000 (placeholder, no km data)',
    ],
    notes='Three separate models, identical hyperparameters, different alpha.\n'
          'Log transform on price improves normality and relative-error behaviour.'
)
print('✓ Slide 07 — Model Architecture')

# ────────────────────────────────────────────────────────────────────────────
# SLIDE 08 — Metrics Summary
# ────────────────────────────────────────────────────────────────────────────
content_slide(
    prs,
    title='Model Performance — Test Set',
    subtitle='80 / 20 stratified split · random_state = 42',
    img_path=fig('05_'),
    notes='MAE reported in € on Q50 predictions (back-transformed from log).\n'
          'Coverage = % of actual prices inside the Q15–Q85 interval.\n'
          'R² computed on log-scale predictions.'
)
print('✓ Slide 08 — Metrics Summary')

# ────────────────────────────────────────────────────────────────────────────
# SLIDE 09 — Actual vs Predicted
# ────────────────────────────────────────────────────────────────────────────
content_slide(
    prs,
    title='Actual vs Predicted Price — Test Set',
    subtitle='2 000 random sample · error bars = Q15–Q85 interval',
    img_path=fig('06_'),
    notes='Points should cluster close to the red dashed diagonal.\n'
          'Error bars show the width of the 70% prediction interval for each vehicle.\n'
          'Some systematic spread visible at high price range — expected with a single model for all brands.'
)
print('✓ Slide 09 — Actual vs Predicted')

# ────────────────────────────────────────────────────────────────────────────
# SLIDE 10 — Residuals
# ────────────────────────────────────────────────────────────────────────────
content_slide(
    prs,
    title='Residual Distribution — Test Set',
    subtitle='Residual = log(predicted Q50) − log(actual)',
    img_path=fig('07_'),
    notes='Near-zero mean residual confirms no systematic bias.\n'
          'Distribution is approximately normal on log scale.\n'
          'Slight right skew visible for under-valued vehicles.'
)
print('✓ Slide 10 — Residuals')

# ────────────────────────────────────────────────────────────────────────────
# SLIDE 11 — SHAP Top-20
# ────────────────────────────────────────────────────────────────────────────
content_slide(
    prs,
    title='Feature Importance — SHAP (Q50 model)',
    subtitle='Top 20 features · TreeExplainer · 50 K test subsample',
    img_path=fig('08_'),
    notes='SHAP dot plot — each point is one vehicle.\n'
          'X-axis: SHAP value (impact on log-price prediction).\n'
          'Colour: feature value (red = high, blue = low).\n'
          'Key drivers: brand/model stats, car_age, mileage proxies.'
)
print('✓ Slide 11 — SHAP')

# ────────────────────────────────────────────────────────────────────────────
# SLIDE 12 — Customs Data Overview
# ────────────────────────────────────────────────────────────────────────────
content_slide(
    prs,
    title="Customs Data — Côte d'Ivoire & Cameroon",
    subtitle='Year distribution vs French training data',
    img_path=fig('09_'),
    notes='Both customs datasets are dominated by older vehicles (2005–2015).\n'
          'Training data peaks around 2018–2022.\n'
          'Domain shift is modest but meaningful — important to flag.'
)
print("✓ Slide 12 — Customs Year Distribution")

# ────────────────────────────────────────────────────────────────────────────
# SLIDE 13 — Descriptive Stats
# ────────────────────────────────────────────────────────────────────────────
content_slide(
    prs,
    title='Customs Data — Descriptive Statistics',
    subtitle="After filtering to known brand-model combinations",
    img_path=fig('10_'),
    notes='Declared prices from VALCAF (CI) and "Valeur imposable" (CM) converted to EUR.\n'
          'CI conversion: VALCAF × 0.0015 EUR\n'
          'CM conversion: XAF × 0.0015 EUR\n'
          'km = 100 000 placeholder for all customs vehicles (no km data available).'
)
print('✓ Slide 13 — Customs Stats')

# ────────────────────────────────────────────────────────────────────────────
# SLIDE 14 — Tier Breakdown
# ────────────────────────────────────────────────────────────────────────────
content_slide(
    prs,
    title='Price Tier Classification — CI vs CM',
    subtitle='GREEN = inside model interval · ORANGE = inside training range · RED = outside',
    img_path=fig('11_'),
    notes='GREEN : declared price within the Q15–Q85 prediction interval → consistent with market\n'
          'ORANGE : outside prediction interval but within historical training min–max → possible undervaluation\n'
          'RED    : completely outside training distribution or no training data → high suspicion'
)
print('✓ Slide 14 — Tier Breakdown')

# ────────────────────────────────────────────────────────────────────────────
# SLIDES 15.x — CI Validation Band Plots (one per brand group)
# ────────────────────────────────────────────────────────────────────────────
ci_validation_figs = sorted(FIG_DIR.glob('12_*.png'))
for vf in ci_validation_figs:
    group_name = vf.stem.replace('12_0', '').replace('12_', '').replace('_validation_ci_', ' ').replace('_', ' ').strip()
    content_slide(
        prs,
        title=f"CI Validation — {group_name.title()}",
        subtitle="Declared price vs model prediction interval & training distribution (year ±3)",
        img_path=vf,
        notes='Blue bar = Q15–Q85 prediction interval. Grey band = training data range (min–max).\n'
              'Coloured dot = declared customs price.\n'
              '[!] = fewer than 5 training samples for that brand-model-year combination.'
    )
    print(f'✓ CI validation slide — {vf.name}')

# ────────────────────────────────────────────────────────────────────────────
# SLIDES 16.x — CM Validation Band Plots (one per brand group)
# ────────────────────────────────────────────────────────────────────────────
cm_validation_figs = sorted(FIG_DIR.glob('13_*.png'))
for vf in cm_validation_figs:
    group_name = vf.stem.replace('13_0', '').replace('13_', '').replace('_validation_cm_', ' ').replace('_', ' ').strip()
    content_slide(
        prs,
        title=f"CM Validation — {group_name.title()}",
        subtitle="Declared price vs model prediction interval & training distribution (year ±3)",
        img_path=vf,
        notes='Same legend as CI slides.'
    )
    print(f'✓ CM validation slide — {vf.name}')

# ────────────────────────────────────────────────────────────────────────────
# FINAL SLIDE — Conclusions
# ────────────────────────────────────────────────────────────────────────────
bullet_slide(
    prs,
    title='Conclusions & Next Steps',
    subtitle='',
    bullets=[
        '✓  LightGBM quantile model provides a robust 70% price corridor for ~620 K French used cars',
        '✓  MAE (Q50) ≈ €1 800 · Coverage Q15–Q85 ≈ 70% on held-out test set',
        '✓  SHAP confirms brand & model stats and car age as top drivers',
        '→  CI & CM declared prices are largely consistent with French market levels',
        '→  RED-tier vehicles (outside training distribution) flagged for manual review',
        '',
        'Next steps:',
        '  • Include mileage data for customs (remove km = 100 000 placeholder)',
        '  • Retrain annually as scrape data updates',
        '  • Extend to additional countries (Morocco, Senegal …)',
    ],
    notes='Q&A slide — leave time for discussion.'
)
print('✓ Final slide — Conclusions')

# ────────────────────────────────────────────────────────────────────────────
# SAVE
# ────────────────────────────────────────────────────────────────────────────
prs.save(OUT_PPTX)
print(f'\n{"─"*60}')
print(f'  ✅  Saved → {OUT_PPTX}')
print(f'  Total slides : {len(prs.slides)}')
print(f'{"─"*60}')
